In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
LOG_DIR = "/content/drive/MyDrive/Research_Logs"

print(f"✅ ログ保存先: {LOG_DIR}")

import huggingface_hub
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(hf_token)

%pip install rank_bm25
%pip install xopen

from pathlib import Path
import sys
import torch
from datasets import load_dataset
from datetime import datetime
import xopen
import random

random.seed(0)


# 以下自作モジュール
module_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/my_modules"
if module_path not in sys.path:
    sys.path.append(module_path)

from ScalingController import ScalingController
from get_model_safe_last_attnweight import get_model_safe, test_patched_model_sanity, remove_cache
from main_inference import main_inference
from run_eval import run_eval_exact_match
from analyze_log import ExperimentAnalyzer

# if 'CACHED_MODEL' not in globals():
#     CACHED_MODEL = None
#     CACHED_TOKENIZER = None

def confirmation():
    """実行前にユーザーの確認を求める"""
    print("\n" + "!" * 30)
    print("警告: 書き出しモードが 'a' (追記) です。")
    print("既存のファイルにデータが追加されますが、よろしいですか？")
    print("!" * 30 + "\n")

    answer = input("実行する場合は 'yes' と入力してください: ").strip().lower()

    if answer != 'yes':
        print("\n[中止] 実行がキャンセルされました。")
        # Jupyterでセルを止める最もクリーンな方法
        raise KeyboardInterrupt("User cancelled the execution.")

    print("\n[開始] 実験を実行します...\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ログ保存先: /content/drive/MyDrive/Research_Logs


In [2]:
from datasets.formatting.formatting import TypeVar
from typing import List, Optional, Type
from pydantic.dataclasses import dataclass

from rank_bm25 import BM25Okapi

PROMPTS_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/lost_in_the_middle/prompts")
T = TypeVar("T")

@dataclass(frozen=True)
class Document:
    title: str
    text: str
    id: Optional[str] = None
    score: Optional[float] = None
    hasanswer: Optional[bool] = None
    isgold: Optional[bool] = None
    original_retrieval_index: Optional[int] = None

    @classmethod
    def from_dict(cls: Type[T], data: dict) -> T:
        data = deepcopy(data)
        if not data:
            raise ValueError("Must provide data for creation of Document from dict.")
        id = data.pop("id", None)
        score = data.pop("score", None)
        isgold = data.pop("isgold", None)
        # Convert score to float if it's provided.
        if score is not None:
            score = float(score)
        return cls(**dict(data, id=id, score=score, isgold=isgold))

def get_qa_prompt(
    question: str, documents: List[Document], mention_random_ordering: bool, query_aware_contextualization: bool
):
    if not question:
        raise ValueError(f"Provided `question` must be truthy, got: {question}")
    if not documents:
        raise ValueError(f"Provided `documents` must be truthy, got: {documents}")

    if mention_random_ordering and query_aware_contextualization:
        raise ValueError("Mentioning random ordering cannot be currently used with query aware contextualization")

    if mention_random_ordering:
        prompt_filename = "qa_ordered_randomly.prompt"
    elif query_aware_contextualization:
        prompt_filename = "qa_with_query_aware_contextualization.prompt"
    else:
        prompt_filename = "qa.prompt"

    with open(PROMPTS_ROOT / prompt_filename) as f:
        prompt_template = f.read().rstrip("\n")

    # Format the documents into strings
    formatted_documents = []
    for document_index, document in enumerate(documents):
        formatted_documents.append(f"Document [{document_index+1}](Title: {document.title}) {document.text}")
    return prompt_template.format(question=question, search_results="\n".join(formatted_documents))

def reorder_bm25(documents, question):
    """
    BM25スコアに基づいてドキュメントを降順に並び替える
    """
    if not documents:
        return []

    # 各ドキュメントを分割、小文字にしてリストで文章を格納
    tokenized_corpus = [doc.text.lower().split() for doc in documents]
    tokenized_query = question.lower().split()

    # bm25
    bm25 = BM25Okapi(tokenized_corpus)
    doc_scores = bm25.get_scores(tokenized_query)

    # スコアの降順（reverse=True）に並び替え
    reordered_docs = [doc for _, doc in sorted(zip(doc_scores, documents), reverse=True, key=lambda x: x[0])]

    return reordered_docs

In [3]:
import torch

def run_inference(
  prompt,
  model,
  tokenizer
):
  inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
  seq_len = inputs.input_ids.shape[1]

  with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.0,
        do_sample=False
    )

  generated_texts = []
  for i in range(inputs.input_ids.shape[0]):
    generated_text = tokenizer.decode(output[i][seq_len:], skip_special_tokens=True)
    generated_texts.append(generated_text.strip())

  return generated_texts

def run_inference_scaled(
  prompt,
  model,
  tokenizer
):
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  seq_len = inputs.input_ids.shape[1]

  scale_map = torch.full((1, seq_len), -1, device=model.device, dtype=torch.float16)
  model.config.current_scale_map = scale_map

  with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.0,
        do_sample=False
    )

  generated_texts = []
  for i in range(inputs.input_ids.shape[0]):
    generated_text = tokenizer.decode(output[i][seq_len:], skip_special_tokens=True)
    generated_texts.append(generated_text.strip())

  return generated_texts

In [4]:
# batched baselineの実装コード

from xopen import xopen
from copy import deepcopy
from tqdm import tqdm
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from itertools import islice

def main():
  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
  tokenizer.padding_side = "left"
  model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        # attn_implementation="eager",　これをつけるとなんかバッチ処理できない
        dtype=torch.float16,
        use_cache=True,
        trust_remote_code=True
  )

  done_count = 0
  if os.path.exists(OUTPUT_PATH):
    with xopen(OUTPUT_PATH, "r") as f:
      done_count = sum(1 for _ in f)

  if done_count >= DATA_NUM:
    print(f"Already processed {done_count} samples.")
    return

  remaining_to_process = DATA_NUM - done_count
  print(f"Resuming from {done_count}. Remaining to process: {remaining_to_process}")

  with xopen(NQ_path, "r") as fin, xopen(OUTPUT_PATH, "a") as fout:
    skipped_fin = islice(fin, done_count, None)
    phar = tqdm(unit=f"sample")

    while remaining_to_process > 0:
      current_batch_size = min(BATCH_SIZE, remaining_to_process)
      lines = list(islice(skipped_fin, current_batch_size))
      if not lines: break

      batch_data = [json.loads(line) for line in lines]
      batch_question = [data["question"] for data in batch_data]
      batch_documents = []
      for data in batch_data:
        documents = []
        for ctx in deepcopy(data["ctxs"]):
          documents.append(Document.from_dict(ctx))
        batch_documents.append(documents)
      batch_documents = [reorder_bm25(documents, question) for question, documents in zip(batch_question, batch_documents)]

      batch_prompt = [get_qa_prompt(question, documents, mention_random_ordering=False, query_aware_contextualization=False) for question, documents in zip(batch_question, batch_documents)]

      results = run_inference(batch_prompt, model, tokenizer)
      # gc.collect()
      # torch.cuda.empty_cache()

      for data, result in zip(batch_data, results):
        output_obj = deepcopy(data)
        output_obj["prediction"] = result
        output_line = json.dumps(output_obj)
        fout.write(output_line + "\n")

      fout.flush()

      processed_count = len(lines)
      remaining_to_process -= processed_count
      phar.update(processed_count)

    phar.close()

In [ ]:
NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_baseline_llama.jsonl"
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

BATCH_SIZE = 8
DATA_NUM = 2655

if __name__ == "__main__":
  main()

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Resuming from 0. Remaining to process: 2655


0sample [00:00, ?sample/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
216sample [04:08,  1.12s/sample]This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (4096). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
2655sample [49:58,  1.13s/sample]


In [4]:
# 既存研究
from xopen import xopen
from copy import deepcopy
from tqdm import tqdm
import json
import os
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from itertools import islice

def main():
    # 安全なロードとパッチ適用（「最後」のコードの特殊処理）
    model, tokenizer = get_model_safe(MODEL_NAME)

    # --- 2. レジューム（再開）処理 ---
    done_count = 0
    if os.path.exists(OUTPUT_PATH):
        with xopen(OUTPUT_PATH, "r") as f:
            done_count = sum(1 for _ in f)
    done_count = VALIDATION_NUM
    if done_count >= DATA_NUM:
        print(f"Already processed {done_count} samples.")
        return

    remaining_to_process = DATA_NUM - done_count
    print(f"Resuming from {done_count}. Remaining to process: {remaining_to_process}")

    # --- 3. メインループ（1件ずつ処理） ---
    with xopen(NQ_path, "r") as fin, xopen(OUTPUT_PATH, "a") as fout:
        # すでに終わっている分をスキップ
        skipped_fin = islice(fin, done_count, None)

        # tqdm の進捗バーを設定
        pbar = tqdm(total=DATA_NUM, initial=done_count, unit="sample")

        # 1件ずつ（current_batch_size = 1）取り出す
        for line in islice(skipped_fin, remaining_to_process):
            data = json.loads(line)

            # ドキュメントの読み込みとDocumentオブジェクト化
            documents = [Document.from_dict(ctx) for ctx in deepcopy(data["ctxs"])]

            # 文書の並び替え（BM25順にしたあと、逆順にする：既存研究ロジック）
            reordered_docs = reorder_bm25(documents, data["question"])

            # プロンプト作成
            prompt = get_qa_prompt(
                data["question"],
                reordered_docs,
                mention_random_ordering=False,
                query_aware_contextualization=False
            )

            # 推論実行（1件ずつのリストとして渡すことでバッチ対応関数をそのまま利用可能）
            # パディングが発生しないため、eagerモードでも正常に動作します
            results = run_inference_scaled([prompt], model, tokenizer)
            result_text = results[0] # 1件なので先頭を取得

            # --- 4. 結果の保存（「最初」の形式を参考） ---
            output_obj = deepcopy(data)
            output_obj["prediction"] = result_text

            fout.write(json.dumps(output_obj) + "\n")
            fout.flush() # 1件ごとに確実に書き出し

            pbar.update(1)

            # 定期的なメモリ解放（必要に応じて）
            # gc.collect()
            # torch.cuda.empty_cache()

        pbar.close()

    print("Processing complete.")

In [ ]:
NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_existing_no_reverse_llama.jsonl"
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

BATCH_SIZE = 1
DATA_NUM = 2655
VALIDATION_NUM = 600

if __name__ == "__main__":
  main()

⏳ Loading new model... (This may take a while)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.
Resuming from 0. Remaining to process: 2655


100%|██████████| 2655/2655 [2:35:12<00:00,  3.51s/sample]

Processing complete.


In [5]:
# batched baselineの実装コード

from xopen import xopen
from copy import deepcopy
from tqdm import tqdm
import json
from transformers import LlamaTokenizer, AutoModelForCausalLM
from itertools import islice

def main():
  tokenizer = LlamaTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
  tokenizer.padding_side = "left"
  model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        # attn_implementation="eager",　これをつけるとなんかバッチ処理できない
        dtype=torch.float16,
        use_cache=True,
        trust_remote_code=True
  )

  done_count = 0
  if os.path.exists(OUTPUT_PATH):
    with xopen(OUTPUT_PATH, "r") as f:
      done_count = sum(1 for _ in f)

  if done_count >= DATA_NUM:
    print(f"Already processed {done_count} samples.")
    return
  done_count += VALIDATION_NUM

  remaining_to_process = DATA_NUM - done_count
  print(f"Resuming from {done_count}. Remaining to process: {remaining_to_process}")

  with xopen(NQ_path, "r") as fin, xopen(OUTPUT_PATH, "a") as fout:
    skipped_fin = islice(fin, done_count, None)
    phar = tqdm(unit=f"sample")

    while remaining_to_process > 0:
      current_batch_size = min(BATCH_SIZE, remaining_to_process)
      lines = list(islice(skipped_fin, current_batch_size))
      if not lines: break

      batch_data = [json.loads(line) for line in lines]
      batch_question = [data["question"] for data in batch_data]
      batch_documents = []
      for data in batch_data:
        documents = []
        for ctx in deepcopy(data["ctxs"]):
          documents.append(Document.from_dict(ctx))
        batch_documents.append(documents)
      batch_documents = [reorder_bm25(documents, question) for question, documents in zip(batch_question, batch_documents)]

      batch_prompt = [get_qa_prompt(question, documents, mention_random_ordering=False, query_aware_contextualization=False) for question, documents in zip(batch_question, batch_documents)]

      results = run_inference(batch_prompt, model, tokenizer)
      # gc.collect()
      # torch.cuda.empty_cache()

      for data, result in zip(batch_data, results):
        output_obj = deepcopy(data)
        output_obj["prediction"] = result
        output_line = json.dumps(output_obj)
        fout.write(output_line + "\n")

      fout.flush()

      processed_count = len(lines)
      remaining_to_process -= processed_count
      phar.update(processed_count)

    phar.close()

In [6]:
NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_baseline_vicuna-16k.jsonl"
MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"

BATCH_SIZE = 8
DATA_NUM = 2655
VALIDATION_NUM = 600

if __name__ == "__main__":
  main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Resuming from 600. Remaining to process: 2055


200sample [03:43,  1.12s/sample]This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (4096). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
2055sample [38:50,  1.13s/sample]


In [4]:
import torch
import sys
from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaTokenizer
from DynamicScalingLlamaAttention_last_attnweight import DynamicScalingLlamaAttention
CACHED_MODEL = None
CACHED_TOKENIZER = None

def get_model_safe(model_path):
    global CACHED_MODEL, CACHED_TOKENIZER

    if CACHED_MODEL is not None:
        print("✅ Model already loaded. Using cached model.")
        return CACHED_MODEL, CACHED_TOKENIZER

    print("⏳ Loading new model... (This may take a while)")

    tokenizer = LlamaTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        attn_implementation="eager",
        dtype=torch.float16,
        use_cache=False,
        trust_remote_code=True
    )

    print("🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...")

    target_layers = list(range(14,20))

    # モデル設定
    model.config.hidden_scale_config = {
        "target_layers": target_layers,
        "target_dims": [2393],
        "factor": -1,
        "last_recompute_tokens": 1,
        "change_value": False
    }

    # 全レイヤー入れ替え
    for i, layer in enumerate(model.model.layers):
        if i in target_layers:
            original_attn = layer.self_attn
            target_device = original_attn.q_proj.weight.device
            target_dtype = original_attn.q_proj.weight.dtype

            new_attn = DynamicScalingLlamaAttention(config=model.config, layer_idx=i)

            # originalの情報をコピー
            new_attn.load_state_dict(original_attn.state_dict(), strict=False)
            new_attn.to(device=target_device, dtype=target_dtype)
            layer.self_attn = new_attn
        else: pass

    CACHED_MODEL = model
    CACHED_TOKENIZER = tokenizer

    print("✅ Model loaded and patched successfully.")
    return CACHED_MODEL, CACHED_TOKENIZER

In [5]:
import torch

def run_inference(
  prompt,
  model,
  tokenizer
):
  inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
  seq_len = inputs.input_ids.shape[1]

  with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.0,
        do_sample=False
    )

  generated_texts = []
  for i in range(inputs.input_ids.shape[0]):
    generated_text = tokenizer.decode(output[i][seq_len:], skip_special_tokens=True)
    generated_texts.append(generated_text.strip())

  return generated_texts

def run_inference_scaled(
  prompt,
  model,
  tokenizer
):
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  seq_len = inputs.input_ids.shape[1]

  scale_map = torch.full((1, seq_len), -1, device=model.device, dtype=torch.float16)
  model.config.current_scale_map = scale_map

  with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.0,
        do_sample=False
    )

  generated_texts = []
  for i in range(inputs.input_ids.shape[0]):
    generated_text = tokenizer.decode(output[i][seq_len:], skip_special_tokens=True)
    generated_texts.append(generated_text.strip())

  return generated_texts

In [6]:
# 既存研究
from xopen import xopen
from copy import deepcopy
from tqdm import tqdm
import json
import os
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from itertools import islice

def main():
    # 安全なロードとパッチ適用（「最後」のコードの特殊処理）
    model, tokenizer = get_model_safe(MODEL_NAME)

    # --- 2. レジューム（再開）処理 ---
    done_count = 0
    if os.path.exists(OUTPUT_PATH):
        with xopen(OUTPUT_PATH, "r") as f:
            done_count = sum(1 for _ in f)

    if done_count >= DATA_NUM:
        print(f"Already processed {done_count} samples.")
        return

    done_count += VALIDATION_NUM

    remaining_to_process = DATA_NUM - done_count
    print(f"Resuming from {done_count}. Remaining to process: {remaining_to_process}")

    # --- 3. メインループ（1件ずつ処理） ---
    with xopen(NQ_path, "r") as fin, xopen(OUTPUT_PATH, "a") as fout:
        # すでに終わっている分をスキップ
        skipped_fin = islice(fin, done_count, None)

        # tqdm の進捗バーを設定
        pbar = tqdm(total=DATA_NUM, initial=done_count, unit="sample")

        # 1件ずつ（current_batch_size = 1）取り出す
        for line in islice(skipped_fin, remaining_to_process):
            data = json.loads(line)

            # ドキュメントの読み込みとDocumentオブジェクト化
            documents = [Document.from_dict(ctx) for ctx in deepcopy(data["ctxs"])]

            # 文書の並び替え（BM25順にしたあと、逆順にする：既存研究ロジック）
            reordered_docs = reorder_bm25(documents, data["question"])

            # プロンプト作成
            prompt = get_qa_prompt(
                data["question"],
                reordered_docs,
                mention_random_ordering=False,
                query_aware_contextualization=False
            )

            # 推論実行（1件ずつのリストとして渡すことでバッチ対応関数をそのまま利用可能）
            # パディングが発生しないため、eagerモードでも正常に動作します
            results = run_inference_scaled([prompt], model, tokenizer)
            result_text = results[0] # 1件なので先頭を取得

            # --- 4. 結果の保存（「最初」の形式を参考） ---
            output_obj = deepcopy(data)
            output_obj["prediction"] = result_text

            fout.write(json.dumps(output_obj) + "\n")
            fout.flush() # 1件ごとに確実に書き出し

            pbar.update(1)

            # 定期的なメモリ解放（必要に応じて）
            # gc.collect()
            # torch.cuda.empty_cache()

        pbar.close()

    print("Processing complete.")

In [7]:
NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_existing_no_reverse_vicuna-16k.jsonl"
MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"

BATCH_SIZE = 1
DATA_NUM = 2655
VALIDATION_NUM = 600

if __name__ == "__main__":
  main()

⏳ Loading new model... (This may take a while)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.
Resuming from 600. Remaining to process: 2055


100%|██████████| 2655/2655 [1:40:07<00:00,  2.92s/sample]

Processing complete.


In [9]:
!wc -l "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_existing_no_reverse_vicuna-16k.jsonl"

2055 /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_existing_no_reverse_vicuna-16k.jsonl


In [ ]:
import time
import gc
import torch
from google.colab import runtime

# 1. メモリの最終開放（念のため）
gc.collect()
torch.cuda.empty_cache()

# 2. 待機設定
# Google Driveの同期には30〜60秒ほど見ておくと非常に安全です
WAIT_TIME_SEC = 60

print(f"✅ 全ての処理が完了しました。")
print(f"📂 Google Driveへの反映を待機しています（{WAIT_TIME_SEC}秒）...")

# プログレスバー付きで待機（あとどれくらいで消えるか視覚化）
from tqdm import tqdm
for _ in tqdm(range(WAIT_TIME_SEC)):
    time.sleep(1)

print("\n🚀 待機完了。セッションを終了し、ランタイムを削除します。")

# 3. セッション終了
runtime.unassign()